# DocCornerNet → TensorFlow.js Converter
Run all cells in order. The final cell creates a ZIP to upload to your server.

In [ ]:
# Step 1: Install dependencies
!pip install tensorflowjs -q
import os, json, zipfile, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
# Step 2: Download model + source code
!wget -q https://raw.githubusercontent.com/mapo80/DocCornerNet-CoordClass/main/model.py -O model.py
!wget -q https://raw.githubusercontent.com/mapo80/DocCornerNet-CoordClass/main/checkpoints/mobilenetv2_224_best/best_model.keras -O best_model.keras
print('Downloaded model.py and best_model.keras')

In [ ]:
# Step 3: Import custom layers and load model
import model
import tensorflow as tf

# Patch SimCCDecode (saved model missing num_bins in config)
orig_init = model.SimCCDecode.__init__
def patched_init(self, **kwargs):
    kwargs.setdefault('num_bins', 224)
    orig_init(self, **kwargs)
model.SimCCDecode.__init__ = patched_init

full_model = tf.keras.models.load_model('best_model.keras', compile=False, safe_mode=False)
print(f'Loaded model. Input: {full_model.input_shape}')
for out in full_model.outputs:
    print(f'  Output: {out.name}, shape: {out.shape}')

In [ ]:
# Step 4: Export full model as SavedModel
# We use the full model (including SimCCDecode) because TF.js Graph model format
# freezes custom layers as standard TF ops → no custom layer issues in the browser
print('Full model output tensors (will appear in TF.js graph model):')
for out in full_model.outputs:
    print(f'  "{out.name}": shape={list(out.shape)}')
print()

In [ ]:
# Step 5: Convert to TF.js Graph model via SavedModel
# Graph models freeze custom layers as standard TF ops → no SimCCDecode issues
# We skip the trimmed model and use the full model
import tensorflowjs as tfjs

output_dir = 'tfjs_graph_model'

# Export full model as SavedModel (preserves all ops including SimCCDecode)
full_model.export('saved_model')

# Convert SavedModel → TF.js Graph model
!tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model \
    --signature_name=serving_default saved_model {output_dir}

# Show output tensors from the signature
imported = tf.saved_model.load('saved_model')
infer = imported.signatures['serving_default']
print('\nGraph outputs (use these in JS with tf.loadGraphModel):')
for name, tensor in infer.structured_outputs.items():
    print(f'  {name}: shape={tensor.shape}')
del imported

print(f'\nConverted! Files in {output_dir}/:')
for fname in os.listdir(output_dir):
    size = os.path.getsize(os.path.join(output_dir, fname))
    print(f'  {fname} ({size:,} bytes)')

In [ ]:
# Step 6: Verify output shapes for JS code
print('Output shapes for app.js (from full model):')
print('---')
for i, out in enumerate(full_model.outputs):
    print(f'Output[{i}] "{out.name}": shape={list(out.shape)}')
print('---')

# Also check the graph model's model.json for output tensor names
import json as _json
with open(os.path.join(output_dir, 'model.json')) as f:
    gm = _json.load(f)
print('\nGraph model.json output signatures:')
for sig_name, sig in gm.get('signatures', {}).items():
    print(f'  Signature "{sig_name}":')
    for out_name, out_info in sig.get('outputs', {}).items():
        print(f'    "{out_name}" → {out_info}')

In [ ]:
# Step 7: Create download ZIP
zip_path = 'tfjs_model.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            fp = os.path.join(root, file)
            zf.write(fp, os.path.join('doccornernet', os.path.relpath(fp, output_dir)))
    zf.writestr('model_info.json', json.dumps({
        'name': 'mobilenetv2_224_best',
        'img_size': 224,
        'num_bins': 224,
        'input_norm': 'imagenet',
        'backbone': 'mobilenetv2_alpha035',
        'outputs': [{'name': out.name, 'shape': list(out.shape)} for out in full_model.outputs]
    }, indent=2))

print(f'Created {zip_path} ({os.path.getsize(zip_path):,} bytes)')
print('\nDownload from Colab file browser (folder icon on left).')
print('Extract and upload doccornernet/ to assets/models/doccornernet/')